In [ ]:
# Import required packages
using DifferentialEquations     # Main ODE solver framework
using LinearAlgebra             # For matrix operations (mul!)
using SparseArrays              # Sparse matrix handling
using KLU                       # Sparse LU factorization
using SparseDiffTools           # For matrix_colors 
using PreallocationTools        # For DiffCache
using DiffEqCallbacks           # For PositiveDomain callback
using SciMLSensitivity, Enzyme  # For automatic differentation
using ADTypes                   # For automatic differentation

using Plots                     # For plotting results
using YAML                      # For parsing mechanism files

using Graphs                    # For network graph structures
using GraphPlot                 # For visualizing graphs

# Include the chemistry module
include("Chemistry.jl")

# Chemistry.jl is at the heart of these simulations, it loads the data from .yaml files, 
# it creates structures and builds a stochiometric matrix and other needed data-objects
# This file also contains the thermodynamic calculations h0 and species_cp
# It is AD supportive... Good enough for AutoEnzyme at least, with the clamp.

"""
Holds the essential, pre-processed chemical data needed for the hot loop.
"""
struct ChemistryParameters
    S::SparseMatrixCSC{Float64, Int64}
    kinetics_list::SplitKinetics  
    species_list::Vector{Species}
end

"""
Holds pre-allocated cache arrays for AD-compatibility and performance.
"""
struct CombustionCache{T}
    r_cache::DiffCache{Vector{T}, Vector{T}}
    dC_dt_cache::DiffCache{Vector{T}, Vector{T}}
    c_p_node_temp_cache::DiffCache{Vector{T}, Vector{T}}
    cp_species_vec_cache::DiffCache{Vector{T}, Vector{T}}
    D_cache::DiffCache{Vector{T}, Vector{T}}
    diffusion_term_cache::DiffCache{Vector{T}, Vector{T}}
    diffusion_term_species_cache::DiffCache{Matrix{T}, Vector{T}}
end
# Note: I added the second type parameter to DiffCache, which is common.
# e.g., DiffCache{Vector{Float64}, Vector{Float64}}

"""
The main parameter object passed to the ODE solver.
"""
struct CombustionParameters{T}
    chem::ChemistryParameters
    A::SparseMatrixCSC{Float64, Int64} # Discretization matrix (e.g., for diffusion)
    V::Vector{Float64}               # Cell volumes
    cache::CombustionCache{T}
end

In [ ]:
# Loading the reaction mechanism

# This map contains several mechanisms: 
# - GRI-Mech 3.0 with 53 species and 325 reactions, filename 'GRI30.yaml'
# - Tianfeng Lu's reduced mechanism for methane, 30 species and 184 reactions, filename 'LuMechanism.yaml'
# - Charlélie Laurent's reduced mechanism for methane, 17 species and 82 reactions, filename 'LaurentMechanism.yaml'
# - A 2 step mechanism by  Franzelli et al., 6 species and 2 reactions, filename '2step.yaml'
# - A 1 step mechanism (unrecommended), 6 species and 1 reaction, filename '1step.yaml'

const FILENAME = "LaurentMechanism.yaml"

# Load the mechanism file
mechanism_data = load_mechanism_data(FILENAME)

# Process the chemical data
species_list, reaction_list, species_index_map = process_chemical_data(mechanism_data)

# Number of species and reactions
const n_species = length(species_list)
const n_vars = n_species + 1
const n_reactions = length(reaction_list)

# Build the stoichiometric matrix and kinetics list
S = sparse(build_stoichiometric_matrix(reaction_list, n_species))
kinetics_list = build_kinetics_list(reaction_list)

# Create chemistry parameters
chem_params = ChemistryParameters(S, kinetics_list, species_list)

println("Chemistry is loaded")

In [ ]:
# Creating the reactor network

# Build network graph
const n_nodes = 12
const T_idx = 1:n_vars:(n_nodes-1)*n_vars+1

g = SimpleGraph(n_nodes)

# Define network topology
edges = [
    (1, 2), (2, 3), (2, 4), (3, 4), (4, 5),
    (5, 6), (5, 7), (6, 7), (7, 8),
    (3, 9), (6,10), (8, 9), (1, 10),
    (8, 11), (9, 11),
    (1, 12), (10, 12),
]

for (i, j) in edges
    add_edge!(g, i, j)
end

# Create connectivity matrix
A = -laplacian_matrix(g)

# Volumes arbitrarily all set to 1
V = ones(n_nodes)

# Set initial conditions by choosing methane nodes
methane_nodes = [4, 7, 9, 12]
hydrogen_nodes = [1, 3, 5]

# Set up the initial state
u0 = zeros(Float64, n_nodes*(n_species + 1))

# Define initial configurations
initial_conditions = Dict(
    :methane => ChemistryConfig(temperature = 1200.0, pressure = 1e6,
                                fuel_mixture = Dict("CH4" => 1/10.57),
                                air_percentage = 9.57/10.57),
    :air     => ChemistryConfig(temperature = 1200.0, pressure = 1e6,
                                fuel_mixture = Dict("CH4" => 0.00),
                                air_percentage = 1.00),
    :hydrogen=> ChemistryConfig(temperature = 1200.0, pressure = 1e6,
                                fuel_mixture = Dict("H2" => 0.99/6.5),
                                air_percentage = 5.5/6.5)
)

# Assign to nodes
for node in 1:n_nodes
    start_idx = (node-1)*n_vars + 1
    end_idx = (node-1)*n_vars + n_vars

    if node in methane_nodes
        config = initial_conditions[:methane]
    elseif node in hydrogen_nodes
        config = initial_conditions[:hydrogen]
    else
        config = initial_conditions[:air]
    end
    u0[start_idx:end_idx] = initialize_concentrations(config, species_list, species_index_map)
end

And now the ODE. I have defined $X$ as a vector that stores the variables as:

$X=(T_1, [A]_1, [B]_1, ... [Z]_1, T_2, [A]_2, ..)$

With the subscripts referring to nodes and $A,B,...$ being species.

The ODE has a main body: system_ode that first iterates over the nodes and calculates $dX$ as the change because of the internal effects of the node by calling reaction_ode_inplace.

The change in concentrations is the product of the stochiometric matrix and the reaction rate vector.

The change in heat is the change in concentrations dot product a vector of enthalpies of species. $dT$ is then computed by dividing by specific heat, which is the molar specific heat of each specie times that species concentration summated over the species.

After the internals there is diffusion, which in general, is a Laplacian-like equation but for graph models we use the graph Laplacian, which this code refers to as $A$. From my derivations I got:

$dT/dt += diag(1/VcP) * L * (k.*T)$

And per specie:

$C/dt += diag(1/V) * L * (D.*C)$

Here $k$ and $D$ are supposed to be vectors of conductivity and diffusivity at each node, but now they are just constants I guessed. Edge weights representing the structure of the connections can be factored into a weighted graph Laplacian, now they are just all 1.

In [ ]:
R=8.141

# === Reaction ODE per node ===

# Reaction ODE per node.
function reaction_ode_inplace!(
    dX::AbstractVector{<:Real},
    X::AbstractVector{<:Real},
    param::ChemistryParameters,
    r::AbstractVector{<:Real},
    dC_dt::AbstractVector{<:Real},
    cv_species_vec::AbstractVector{<:Real}
)
    T_val = X[1]
    
    # --- 1. Compute Reaction Rates ---
    
    compute_reaction_rates!(r, X, param.kinetics_list, param.species_list)

    # --- 2. Compute Species Source Terms (dC/dt) ---
    # This uses the pre-computed stoichiometric matrix S.
    mul!(dC_dt, param.S, r)
    dX[2:end] = dC_dt

    # --- 3. Compute Temperature Change (dT/dt) ---
    concentrations = X[2:end]

    # Optimized volumetric heat capacity (unchanged, this was already good)
    for i in eachindex(concentrations)
        # species_cp() is your function for species heat capacity
        cv_species_vec[i] = species_cp(T_val, param.species_list[i].thermo) - R_joule
    end
    cv_vol = dot(concentrations, cv_species_vec)

    # Reuse cp_species_vec as a vector of entalphies of species
    for i in eachindex(concentrations)
        # species_cp() is your function for species heat capacity
        cv_species_vec[i] = h0(T_val, param.species_list[i].thermo) - R_joule * T_val
    end
     
    total_heat = -dot(cv_species_vec, dC_dt)

    # dT = dH / c_P (These are all intrinsic quantities: /m3)
    dX[1] = total_heat / cv_vol

    # Return the volumetric heat capacity for the diffusion term calculation
    return cv_vol
end

# === Main system ODE ===
function system_ode!(du::AbstractVector{<:Real}, u::AbstractVector{<:Real}, p, t::Real)
    # Unpack parameters and caches (this part is perfect)
    chem, A, V = p.chem, p.A, p.V
    cache = p.cache
    r = get_tmp(cache.r_cache, u)
    dC_dt = get_tmp(cache.dC_dt_cache, u)
    D = get_tmp(cache.D_cache, u)
    c_p_node_temp = get_tmp(cache.c_p_node_temp_cache, u)
    cp_species_vec = get_tmp(cache.cp_species_vec_cache, u)

    u = max.(u, 1e-100)  # This creates an artifact that slowly heats the system
                         # But the alternative makes implicit methods unstable

    # === 1. Reaction calculations (per-node) ===
    for i in 1:n_nodes
        node_idx = (i-1)*n_vars + 1: i*n_vars
        @views c_p_node_temp[i] = reaction_ode_inplace!(du[node_idx] , u[node_idx], chem, r, dC_dt, cp_species_vec)
    end
    
    # === 2. Diffusion calculations (vectorized) ===
    invV = Diagonal(1.0 ./ V)
    invc_p = Diagonal(1.0 ./ (V .* c_p_node_temp))


    # Temperature diffusion: dC/dt += (1/VcP) * L * (k*T)
    du[T_idx] += invc_p * A * (conductivity.(u[T_idx]) .* u[T_idx])
    
    # Species diffusion: dC/dt += (1/V) * L * (D*C)
    for i in 1:n_species
        for node in 1:n_nodes
            start_idx = (node-1)*n_vars + 2
            end_idx = (node-1)*n_vars + n_vars
            P = sum(u[start_idx:end_idx]) * u[(node-1)*n_vars+1] * R
            D[node]=diffusivity(chem.species_list[i].trans, chem.species_list[species_index_map["N2"]].trans, u[(node-1)*n_vars+1], P)
        end
        du[T_idx .+ i] += invV * A * (D .* u[T_idx .+ i])
    end

    return nothing
end

In [ ]:
# === Jacobian Sparsity ===
# I analytically derive Jacobian sparsity patterns

function build_local_jacobian_sparsity(param::ChemistryParameters)
    stoich_mat = param.S
    J = spzeros(Bool, n_vars, n_vars)
    for r in 1:size(stoich_mat, 2)
        participants = findall(!iszero, stoich_mat[:, r])
        for i in participants, j in participants
            J[i+1, j+1] = true
        end
    end
    for spec in 1:n_species
        J[spec+1, 1] = true
        J[1, spec+1] = true
    end
    J[1, 1] = true
    return J
end

function build_network_jacobian_sparsity(param::ChemistryParameters, laplacian::SparseMatrixCSC)
    local_jac = build_local_jacobian_sparsity(param)
    n_total = n_vars * n_nodes
    J_network = spzeros(Bool, n_total, n_total)
    for node in 1:n_nodes
        idx_start = (node - 1) * n_vars + 1
        idx_end = node * n_vars
        idx_range = idx_start:idx_end
        J_network[idx_range, idx_range] = local_jac
    end
    lap_rows, lap_cols, _ = findnz(laplacian)
    for (i, j) in zip(lap_rows, lap_cols)
        if i != j
            for var in 1:n_vars
                row_idx = (i - 1) * n_vars + var
                col_idx = (j - 1) * n_vars + var
                J_network[row_idx, col_idx] = true
            end
        end
    end
    return J_network
end

jac_sparsity = build_network_jacobian_sparsity(chem_params, A)
colorvec = matrix_colors(jac_sparsity)

# Determine the chunk size
chunk_size = maximum(colorvec)

In [ ]:
# === System Initialization ===

# Create the parameter object `p` as a NamedTuple

# Function to build the cache
function build_cache(T::Type, n_species, n_reactions, n_nodes, chunk_size)
    # The chunk size is passed to the DiffCache constructor.
    # The second type parameter (e.g., Vector{T}) is what get_tmp will use.
    return CombustionCache{T}(
        DiffCache(zeros(T, n_reactions), chunk_size),          # r_cache
        DiffCache(zeros(T, n_species), chunk_size),            # dC_dt_cache
        DiffCache(zeros(T, n_nodes), chunk_size),              # c_p_node_temp_cache
        DiffCache(zeros(T, n_species), chunk_size),            # cp_species_vec_cache
        DiffCache(zeros(T, n_nodes), chunk_size),              # diffusivity cache
        DiffCache(zeros(T, n_nodes), chunk_size),              # diffusion_term_cache
        DiffCache(zeros(T, n_nodes, n_species), chunk_size)    # diffusion_term_species_cache
    )
end

p = CombustionParameters(
    chem_params,
    float.(A),
    V,
    build_cache(Float64, n_species, n_reactions, n_nodes, chunk_size)
)

ode_func = ODEFunction(system_ode!, sparsity=jac_sparsity, colorvec=colorvec)

println("ready to go!")

In [ ]:
# === A Progress Printing Callback ===

const PROGRESS_PRINT_ITER_INTERVAL = 10000

# This Ref will hold the next iteration at which we want to print.
# It will be initialized correctly at the start of each solve.
const next_print_iter_ref = Ref(0) # Changed to integer type for iterations

# 1. Condition function: Checks if the current iteration has reached the next scheduled print iteration
function condition_iter_progress(u, t, integrator)
    # Trigger if current iteration is >= next scheduled iteration
    return integrator.iter >= next_print_iter_ref[]
end

# 2. Affect function: Executes when the condition is true
function affect_iter_progress!(integrator)
    # Update the next scheduled print iteration.
    # Ensures we print at the desired interval, even if the solver does more than 1 iter between triggers.
    next_print_iter_ref[] = floor(Int, integrator.iter / PROGRESS_PRINT_ITER_INTERVAL + 1) * PROGRESS_PRINT_ITER_INTERVAL

    # Extract relevant info from the integrator
    u = integrator.u
    
    # Get values and format simply
    current_iter = integrator.iter # The current iteration count
    t_ns = integrator.t    # Current time in nanoseconds
    temps = u[T_idx]        # Assuming Temperature is the first variable (index 1) in each node
    dt_ps = integrator.dt  # Timestep in picoseconds

    ch4_con= u[T_idx.+species_index_map["CH4"]]
    co_con = u[T_idx.+species_index_map["CO"]]

    println("Iter= $(current_iter), t= $(round(t_ns, digits=14)) s, dt = $(round(dt_ps, digits=18)) s")
    println("Temperature range: Tmin = $(round(minimum(temps), digits = 3)), Tmax = $(round(maximum(temps), digits = 3))")
    println("Total CH4 concentration = $(round(sum(ch4_con), digits = 2)), Total CO concentration = $(round(sum(co_con), digits = 2))")

    # Optional warning for very small timesteps
    if abs(integrator.dt) < 1e-21
        println("  WARNING: Extremely small timestep (< 1e-21 s) detected.")
    end
end

# 3. Initialization function: Sets up next_print_iter_ref at the start of each solve
function initialize_iter_progress!(cb, u, t, integrator)
    # Start printing from the first multiple of PROGRESS_PRINT_ITER_INTERVAL
    # after the initial iteration (which is 1)
    next_print_iter_ref[] = 1
end

# Create the progress callback instance
cb1 = DiscreteCallback(
    condition_iter_progress,
    affect_iter_progress!,
    initialize=initialize_iter_progress!,
    save_positions=(false, false) # Don't save positions from this callback
)

cb2 = PositiveDomain(; save = false)

cb = CallbackSet(cb1,cb2)

The problem is so stiff that I am solving it in phases. Phase 1 does the hydrogen combustion, and it is not a problem.

In [ ]:
# === solve(problem) I ===
# Very stiff so we use a stronger method which handles
# explosion stiffness and its discontinuities well

tspan = (0.0, 1e-5)

problem = ODEProblem(ode_func, u0, tspan, p)

algo = AutoVern7(KenCarp47(
    linsolve = KLUFactorization(),
    autodiff = AutoEnzyme(; function_annotation=Enzyme.Duplicated),
    standardtag = false,
    concrete_jac = true
))

println("Beginning integration using $FILENAME")
println("  Time span: $(tspan)")

@time sol1 = solve(problem, algo;
    abstol=1e-12, reltol=1e-9,   # Extreme tolerances improve stability
    callback = cb,
    save_everystep=false,
    saveat=1e-7
)

In [ ]:
# === Print Results I ===

times1 = sol1.t
n_times1 = length(times1)

time_series1 = Array{Float64}(undef, n_times1, n_nodes, n_vars)

for (i, u_vec) in enumerate(sol1.u)
    for k in 1:n_nodes
        id_begin= (k-1)*n_vars + 1
        id_end= k*n_vars
        time_series1[i,k,:]=u_vec[id_begin:id_end]
    end
end


tvals = sol1.t
idx = findall(t -> t <= 100, tvals)

plot1 = plot(times1[idx], time_series1[idx, :, 1], label=["Node $i" for i in (1:n_nodes)'],
     xlabel="t (s)", ylabel="T (K)", legend=:right)

println(maximum(time_series1[:,:,1]))

display(plot1)
savefig("LaurentnetworkT1")

idx = findall(t -> t <= 100, tvals)

H2_idx = species_index_map["H2"]
CO_idx = species_index_map["OH"]
O2_idx = species_index_map["O2"]
H2O_idx = species_index_map["H2O"]
total_H2=[]
total_H2O=[]
total_CO=[]
total_O2=[]

for i in 1:n_times1
    push!(total_H2, sum(time_series1[i, :, H2_idx + 1]))  # +1 because T is index 1
    push!(total_H2O, sum(time_series1[i, :, H2O_idx + 1]))  # +1 because T is index 1
    push!(total_CO, sum(time_series1[i, :, CO_idx + 1]))  # +1 because T is index 1
    push!(total_O2, sum(time_series1[i, :, O2_idx + 1]))  # +1 because T is index 1
end

plot2 = plot(times1[idx], vec(total_H2)[idx], xlabel="t (s)", ylabel="n (mol)",
     label="Total H₂ Over Time")
plot!(plot2, times1[idx], vec(total_H2O)[idx], xlabel="t (s)", ylabel="n (mol)",
     label="Total H₂O Over Time")
plot!(plot2, times1[idx], vec(total_CO)[idx], xlabel="t (s)", ylabel="n (mol)",
     label="Total OH Over Time")
plot!(plot2, times1[idx], vec(total_O2)[idx], xlabel="t (s)", ylabel="n (mol)",
     label="Total O₂ Over Time")
display(plot2)
savefig("Laurentnetworktotals1")

In [ ]:
# === solve(problem) II ===
# More stiff as we enter timescales wirh many active processes at once

tspan = (sol1.t[end], 0.0005)

problem = ODEProblem(ode_func, sol1.u[end], tspan, p)

algo = AutoVern7(KenCarp3(
    linsolve = KLUFactorization(),
    autodiff = AutoEnzyme(; function_annotation=Enzyme.Duplicated),
    standardtag = false,
    concrete_jac = true
))

println("Beginning integration using $algo with dataset $FILENAME")
println("  Time span: $(tspan)")

# Problem becomes unstable at too loose tolerance settings

@time sol2 = solve(problem, algo;
    abstol=1e-12, reltol=1e-9, 
    save_everystep=false, saveat=1e-6, 
    callback = cb)

In [ ]:
# === Plot Results II ===

times2 = sol2.t
n_times2 = length(times2)

time_series2 = Array{Float64}(undef, n_times2, n_nodes, n_vars)


for (i, u_vec) in enumerate(sol2.u)
    for k in 1:n_nodes
        id_begin= (k-1)*n_vars + 1
        id_end= k*n_vars
        time_series2[i,k,:]=u_vec[id_begin:id_end]
    end
end

great_times = vcat(times1,times2)
great_time_series = vcat(time_series1,time_series2)


plot1 = plot(great_times, great_time_series[:,:, 1], label=["Node $i" for i in (1:n_nodes)'],
     xlabel="t (s)", ylabel="T (K)")

println(maximum(great_time_series[:,:,1]))

display(plot1)
savefig("LaurentnetworkT2")

CH4_idx = species_index_map["CH4"]
CO_idx = species_index_map["CO"]
CO2_idx = species_index_map["CO2"]
O2_idx = species_index_map["O2"]
total_CH4=[]
total_CO=[]
total_CO2=[]
total_O2=[]


for i in 1:length(great_times)
    push!(total_CH4, sum(great_time_series[i, :, CH4_idx + 1]))  # +1 because T is index 1
    push!(total_CO, sum(great_time_series[i, :, CO_idx + 1]))  # +1 because T is index 1
    push!(total_CO2, sum(great_time_series[i, :, CO2_idx + 1]))  # +1 because T is index 1
    push!(total_O2, sum(great_time_series[i, :, O2_idx + 1]))  # +1 because T is index 1
end
    
println(size(total_CH4))
plot2 = plot(great_times, total_CH4, xlabel="t (s)", ylabel="n (mol)",
     label="Total CH₄ Over Time")
plot!(plot2, great_times, total_CO, xlabel="t (s)", ylabel="n (mol)",
     label="Total CO Over Time")
plot!(plot2, great_times, total_CO2, xlabel="t (s)", ylabel="n (mol)",
     label="Total CO₂ Over Time")
plot!(plot2, great_times, total_O2, xlabel="t (s)", ylabel="n (mol)",
     label="Total O₂ Over Time")
display(plot2)
savefig("Laurentnetworktotals2")

In [ ]:
# === solve(problem) III ===
# Not so stiff so we use an fast method that solves
# stable systems quicker than previous solvers.

tspan = (sol2.t[end], 10)

problem = ODEProblem(ode_func, sol2.u[end], tspan, p)

# Switch to multistep method because no more discontinuities expected
algo = FBDF(
    linsolve = KLUFactorization(),
    autodiff = AutoEnzyme(; function_annotation=Enzyme.Duplicated),
    standardtag = false,
    concrete_jac = true
)

println("Beginning integration using $algo with dataset $FILENAME")
println("  Time span: $(tspan)")

@time sol3 = solve(problem, algo;
    save_everystep = false,
    saveat = 0.001,  # Can use larger save intervals
    maxiters = 1e7,
    reltol = 1e-4, 
    abstol = 1e-6,
    callback = cb
)

In [ ]:
times3 = sol3.t
n_times3 = length(times3)


time_series3 = Array{Float64}(undef, n_times3, n_nodes, n_vars)


for (i, u_vec) in enumerate(sol3.u)
    for k in 1:n_nodes
        id_begin= (k-1)*n_vars + 1
        id_end= k*n_vars
        time_series3[i,k,:]=u_vec[id_begin:id_end]
    end
end

great_times = vcat(times1,times2,times3)
great_time_series = vcat(time_series1,time_series2,time_series3)

plot1 = plot(great_times, great_time_series[:, :, 1], label=["Node $i" for i in (1:n_nodes)'],
     xlabel="t (s)", ylabel="T (K)",legend=:left)

println(maximum(great_time_series[:,:,1]))

display(plot1)
savefig("LaurentnetworkT3")

plot1 = plot(great_times, great_time_series[:, :, species_index_map["CO"]], label=["Node $i" for i in (1:n_nodes)'],
     xlabel="t (s)", ylabel="[CO] (mol/m³)", legend=:left)

display(plot1)
savefig("LaurentnetworkCO")

In [ ]:
using LinearAlgebra: norm

function compute_relaxation_time(sol; tol=1e-6)
    """
    Compute relaxation time τ for a solution `sol` of an ODEProblem.
    
    τ is defined as the time when ‖u(t) - u_steady‖ / ‖u0 - u_steady‖ ≈ 1/e.
    
    Args:
        sol: Solution object from DifferentialEquations.jl
        tol: Tolerance for checking convergence (default: 1e-6)
    
    Returns:
        τ: Relaxation time (first time when decay reaches 1/e)
        If no such time is found, returns `nothing`.
    """
    u0 = sol.prob.u0          # Initial (perturbed) state
    u_steady = sol[end]       # Equilibrium state (final value)
    Δ0 = norm(u0 - u_steady)  # Initial deviation magnitude
    
    # Handle cases where Δ0 ≈ 0 (no perturbation)
    if Δ0 < tol
        @warn "Initial state is already at equilibrium (‖Δu‖ = $Δ0). τ is undefined."
        return nothing
    end
    
    # Target deviation: Δ0 / e
    target_deviation = Δ0 / MathConstants.e
    
    # Find the first time when ‖u(t) - u_steady‖ ≤ target_deviation
    τ = nothing
    for (i, t) in enumerate(sol.t)
        Δu = norm(sol.u[i] - u_steady)
        if Δu ≤ target_deviation + tol  # Allow numerical tolerance
            τ = t
            break
        end
    end
    
    if τ === nothing
        @warn "No relaxation time found within solution timeframe. Increase `tspan`."
    end
    
    return τ
end

compute_relaxation_time(sol1),compute_relaxation_time(sol2),compute_relaxation_time(sol3)